# 07 · Results Analysis

Aggregate and compare experiment runs, build report tables and figures, and show how to log a run. This is where dissertation figures come from.

- **Inputs:** `experiments/*.csv` (+ illustrative demo rows)
- **Outputs:** Comparison tables and a figure exported to `reports/figures/`.

> ⚠️ **Sample vs. real data.** This notebook runs on the committed 10-row synthetic sample so the toolchain works without PRismBench. The sample has singleton classes, so metrics here are *illustrative only*. Each `TODO` marks where the real dataset in `data/raw/` plugs in.

In [ ]:
# --- Standard setup: locate project root, add src/ to path, load helpers ---
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    """Walk upwards until we find the repo root (has pyproject.toml + src/pr_risk)."""
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "src" / "pr_risk").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 50)
SAMPLE_CSV = PROJECT_ROOT / "data" / "sample" / "sample_prs.csv"
print("Project root :", PROJECT_ROOT)
print("Sample CSV   :", SAMPLE_CSV.name, "| exists:", SAMPLE_CSV.exists())

In [ ]:
import matplotlib.pyplot as plt

try:
    import seaborn as sns

    sns.set_theme(style="whitegrid")
    HAS_SNS = True
except ImportError:  # seaborn is optional; matplotlib is enough
    HAS_SNS = False
print("seaborn available:", HAS_SNS)

## 1. Load the experiment logs
The tracking CSVs ship as headers only; we read them, then use demo rows to illustrate analysis.

In [ ]:
from pr_risk.data.load_data import load_csv

log_path = PROJECT_ROOT / "experiments" / "experiment_log.csv"
log = load_csv(log_path)
print("logged runs:", len(log))
print("columns:", list(log.columns))

## 2. Model comparison (demo)
Replace this demo frame with real rows once you have logged runs.

In [ ]:
demo = pd.DataFrame([
    {"model": "logistic_regression", "accuracy": 0.78, "f1_score": 0.76, "roc_auc": 0.81},
    {"model": "random_forest",       "accuracy": 0.83, "f1_score": 0.82, "roc_auc": 0.88},
    {"model": "xgboost",             "accuracy": 0.85, "f1_score": 0.84, "roc_auc": 0.90},
    {"model": "codebert",            "accuracy": 0.88, "f1_score": 0.87, "roc_auc": 0.92},
]).set_index("model")
display(demo)

## 3. Comparison figure → `reports/figures/`

In [ ]:
ax = demo[["accuracy", "f1_score", "roc_auc"]].plot(
    kind="bar", figsize=(8, 4), title="Model comparison (illustrative)"
)
ax.set_ylabel("score")
ax.set_ylim(0, 1)
plt.tight_layout()

fig_dir = PROJECT_ROOT / "reports" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / "model_comparison_demo.png"
plt.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print("saved:", fig_path.relative_to(PROJECT_ROOT))

## 4. Active-learning curve (demo)
Accuracy vs. labelled budget per strategy — the headline AL result for **RQ3**.

In [ ]:
al_demo = pd.DataFrame({
    "labelled": [20, 40, 60, 80, 100],
    "random": [0.62, 0.66, 0.70, 0.73, 0.75],
    "least_confidence": [0.62, 0.71, 0.78, 0.82, 0.84],
    "margin": [0.62, 0.72, 0.80, 0.84, 0.86],
}).set_index("labelled")
display(al_demo)
al_demo.plot(marker="o", figsize=(7, 4), title="Active-learning curves (illustrative)")
plt.xlabel("labelled examples")
plt.ylabel("accuracy")
plt.tight_layout()
plt.show()

## 5. Log a run
Append rows with `scripts/create_sample_experiment.py`, or in code (shown without writing to disk).

In [ ]:
new_row = {
    "experiment_id": "demo-001", "date": "2026-06-24", "dataset_version": "sample",
    "model_name": "random_forest", "task_type": "binary_risk_prediction", "target": "is_risky",
    "active_learning_strategy": "none", "active_learning_round": 0, "labelled_sample_count": 7,
    "accuracy": 0.83, "precision": 0.82, "recall": 0.83, "f1_score": 0.82, "roc_auc": 0.88,
    "notes": "demo row (not written to disk)",
}
preview = pd.concat([log, pd.DataFrame([new_row])], ignore_index=True)
print("To persist for real, run:  python scripts/create_sample_experiment.py")
preview.tail(3)

## Next steps / TODO (real data)
- Populate `experiments/experiment_log.csv` from real runs (nb 03–05) and curate `results_summary.csv`.
- Replace demo frames with real metrics; export final tables to `reports/tables/` and figures to `reports/figures/`.
- Build the final model comparison in `reports/model_comparison/` for the dissertation.